# Week 3 — tourist DP playground

Open in VS Code. **Select Kernel** (Python 3), then run cells from the top (needs `matplotlib`).

Same $3\times 3$ city as the lecture. You will:
1. draw the city,
2. walk **greedily** (expect score $8$),
3. watch the **DP table** fill (sink $=9$),
4. **traceback** the path,
5. compare, then feel why listing paths does not scale.


## Setup — lecture weights

`east[i][j]` / `south[i][j]` = score of the edge **into** cell $(i,j)$.


In [ ]:
import math
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output

# --- lecture city (do not change until the "play" cell) ---
east = [
    [0, 1, 3],
    [0, 2, 1],
    [0, 4, 2],
]
south = [
    [0, 0, 0],
    [2, 3, 1],
    [1, 2, 4],
]
n = m = 3
DRAW_EVERY = 1   # redraw every filled cell (set 2 to speed up)
PAUSE = 0.25


## Draw the city

Hotel top-left, cafe bottom-right. Blue = east edges, terracotta = south.


In [ ]:
def draw_city(ax, path=None, title="tourist city"):
    """path: list of (i,j) cells from hotel to cafe, or None."""
    ax.clear()
    # intersections
    for i in range(n):
        for j in range(m):
            ax.scatter(j, -i, s=120, c="#1c2430" if (i, j) in [(0, 0), (n - 1, m - 1)] else "#2a5a8c", zorder=3)
    ax.text(0, 0.18, "H", ha="center", fontsize=11, color="#b56a3a")
    ax.text(m - 1, -(n - 1) - 0.28, "C", ha="center", fontsize=11, color="#b56a3a")
    # east edges
    for i in range(n):
        for j in range(1, m):
            ax.annotate("", xy=(j, -i), xytext=(j - 1, -i),
                        arrowprops=dict(arrowstyle="->", color="#2a5a8c", lw=1.8))
            ax.text(j - 0.5, -i + 0.18, str(east[i][j]), color="#2a5a8c", ha="center", fontsize=12, fontweight="bold")
    # south edges
    for i in range(1, n):
        for j in range(m):
            ax.annotate("", xy=(j, -i), xytext=(j, -(i - 1)),
                        arrowprops=dict(arrowstyle="->", color="#b56a3a", lw=1.8))
            ax.text(j - 0.22, -(i - 0.5), str(south[i][j]), color="#b56a3a", ha="center", fontsize=12, fontweight="bold")
    if path and len(path) > 1:
        xs = [j for i, j in path]
        ys = [-i for i, j in path]
        ax.plot(xs, ys, "-", color="#b56a3a", lw=3.5, zorder=2, alpha=0.85)
    ax.set_xlim(-0.6, m - 0.4)
    ax.set_ylim(-(n - 1) - 0.5, 0.5)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)

fig, ax = plt.subplots(figsize=(5, 4.5))
draw_city(ax)
plt.show()


## Greedy walker

At each intersection, take the higher of the available south / east street (when both exist). On this city that is **SESE** and scores **8**.


In [ ]:
def greedy_walk(east, south):
    i = j = 0
    path = [(0, 0)]
    score = 0
    moves = []
    while i < n - 1 or j < m - 1:
        can_s = i < n - 1
        can_e = j < m - 1
        if can_s and can_e:
            if south[i + 1][j] >= east[i][j + 1]:
                score += south[i + 1][j]
                i += 1
                moves.append("S")
            else:
                score += east[i][j + 1]
                j += 1
                moves.append("E")
        elif can_s:
            score += south[i + 1][j]
            i += 1
            moves.append("S")
        else:
            score += east[i][j + 1]
            j += 1
            moves.append("E")
        path.append((i, j))
    return path, score, "".join(moves)

g_path, g_score, g_moves = greedy_walk(east, south)
fig, ax = plt.subplots(figsize=(5, 4.5))
draw_city(ax, path=g_path, title=f"greedy {g_moves}  score = {g_score}")
plt.show()
print(f"greedy path {g_path}  moves {g_moves}  score {g_score}")
assert g_score == 8, "expected lecture greedy score 8"


## DP fill — watch the table

Fill best-to-every-cell. Pointers `D` remember north vs west. Sink should be **9**.


In [ ]:
def draw_S(ax, S, highlight=None, title="S"):
    ax.clear()
    for i in range(n):
        for j in range(m):
            val = S[i][j]
            face = "#f7f3ee" if highlight == (i, j) else "#eef3f8"
            ax.add_patch(plt.Rectangle((j - 0.45, -i - 0.45), 0.9, 0.9,
                                       facecolor=face, edgecolor="#c5ccd6", lw=2))
            if val is None:
                txt = "—"
            elif val == float("-inf"):
                txt = "NA"
            else:
                txt = str(val)
            ax.text(j, -i, txt, ha="center", va="center", fontsize=16, fontweight="bold", color="#1c2430")
    ax.set_xlim(-0.7, m - 0.3)
    ax.set_ylim(-(n - 1) - 0.7, 0.7)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)

def fill_dp(east, south, animate=True, blocked=None):
    """Return S, D. blocked = set of (i,j) forbidden (not source/sink)."""
    S = [[None] * m for _ in range(n)]
    D = [[None] * m for _ in range(n)]
    S[0][0] = 0
    step = 0
    def show(hi=None):
        nonlocal step
        step += 1
        if not animate:
            return
        if step % DRAW_EVERY != 0 and hi != (n - 1, m - 1):
            return
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(4.5, 4))
        draw_S(ax, S, highlight=hi, title=f"filling S  (sink so far: {S[n-1][m-1]})")
        plt.show()
        time.sleep(PAUSE)

    show((0, 0))
    for j in range(1, m):
        if blocked and (0, j) in blocked:
            S[0][j] = float("-inf")
        else:
            S[0][j] = S[0][j - 1] + east[0][j]
            D[0][j] = "W"
        show((0, j))
    for i in range(1, n):
        if blocked and (i, 0) in blocked:
            S[i][0] = float("-inf")
        else:
            S[i][0] = S[i - 1][0] + south[i][0]
            D[i][0] = "N"
        show((i, 0))
    for i in range(1, n):
        for j in range(1, m):
            if blocked and (i, j) in blocked:
                S[i][j] = float("-inf")
                show((i, j))
                continue
            a = S[i - 1][j] + south[i][j]
            b = S[i][j - 1] + east[i][j]
            if a >= b:
                S[i][j] = a
                D[i][j] = "N"
            else:
                S[i][j] = b
                D[i][j] = "W"
            show((i, j))
    return S, D

S, D = fill_dp(east, south, animate=True)
print("S =")
for row in S:
    print(row)
print("sink", S[n - 1][m - 1])
assert S[2][2] == 9


## Traceback

Walk pointers from the cafe back to the hotel; reverse to get the path.


In [ ]:
def traceback(D):
    i, j = n - 1, m - 1
    path = [(i, j)]
    while i > 0 or j > 0:
        if D[i][j] == "N":
            i -= 1
        elif D[i][j] == "W":
            j -= 1
        else:
            break
        path.append((i, j))
    path.reverse()
    return path

def path_score(path, east, south):
    sc = 0
    for (i0, j0), (i1, j1) in zip(path, path[1:]):
        if i1 == i0 + 1 and j1 == j0:
            sc += south[i1][j1]
        elif j1 == j0 + 1 and i1 == i0:
            sc += east[i1][j1]
        else:
            raise ValueError("illegal step")
    return sc

dp_path = traceback(D)
dp_score = path_score(dp_path, east, south)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
draw_city(axes[0], path=dp_path, title=f"DP path  score = {dp_score}")
draw_S(axes[1], S, title=f"S  sink = {S[n-1][m-1]}")
plt.tight_layout()
plt.show()
print("DP path", dp_path, "score", dp_score)
assert dp_score == S[n - 1][m - 1]


## Compare greedy vs DP


In [ ]:
print(f"greedy: {g_moves:6s}  score {g_score}")
print(f"DP:     path {dp_path}  score {dp_score}")
print("Greedy took the higher street at every corner — and still lost.")


## Scale: listing paths vs filling a table

On an $n\times m$ step grid there are $\binom{n+m}{n}$ south–east walks. The DP table has $nm$ cells.


In [ ]:
N = 20
print(f"paths for {N}x{N} steps:  C({2*N},{N}) = {math.comb(2*N, N):,}")
print(f"DP cells for {N}x{N} grid: {N*N}")

# tiny timing: fill a random 8x8
import random
random.seed(0)
N2 = 8
e2 = [[0 if j == 0 else random.randint(1, 9) for j in range(N2)] for _ in range(N2)]
s2 = [[0 if i == 0 else random.randint(1, 9) for j in range(N2)] for i in range(N2)]
t0 = time.perf_counter()
# inline fill without animation
SS = [[0] * N2 for _ in range(N2)]
for j in range(1, N2):
    SS[0][j] = SS[0][j - 1] + e2[0][j]
for i in range(1, N2):
    SS[i][0] = SS[i - 1][0] + s2[i][0]
for i in range(1, N2):
    for j in range(1, N2):
        SS[i][j] = max(SS[i - 1][j] + s2[i][j], SS[i][j - 1] + e2[i][j])
print(f"filled {N2}x{N2} in {1000*(time.perf_counter()-t0):.3f} ms  sink={SS[-1][-1]}")


## Optional — blocked intersection

Close one cell (not hotel/cafe). Refill; the path must go around.


In [ ]:
blocked = {(1, 1)}  # try (1,0) or (0,1) too
S2, D2 = fill_dp(east, south, animate=False, blocked=blocked)
p2 = traceback(D2)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
draw_city(axes[0], path=p2, title=f"blocked {blocked}  score = {S2[n-1][m-1]}")
draw_S(axes[1], S2, title="S with NA cell")
plt.tight_layout()
plt.show()
print("path", p2, "sink", S2[n - 1][m - 1])


## Play

Bump a street score (e.g. `east[1][2] = 10`) in the setup cell, re-run greedy + DP. Does the winner change?

## Questions

**Q1.** Why is filling $S$ cheaper than listing all south–east paths?

A1.

**Q2.** What does one cell store, and who are its two predecessors?

A2.

**Q3.** We only want **one** answer — the best path to the sink (or only $F_n$). Why do we still fill a table for **every** cell (or every $F_k$)?

A3.
